In [1]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model
from langchain_ollama import ChatOllama
import sqlite3
import logging

from osint_agent.retrieval.insert_data import (
    DB_PATH
    DOC_TABLE
    ENT_TABLE
)

LOGGER = logging.getLogger(__name__)


model = ChatOllama(
    model="llama3.2",
    temperature=0
)


# Define tools
@tool
def search_documents(query: str) -> dict: #TODO Create search_documents tool
    """Search stored relevant OSINT documents and entities for records from user query.
    1. Normalize the user query
    2. Extract useful search terms
    3. Query the database
    4. Return structured results

    Args:
        query: a user string query
    """

    normalized_query = query.strip().lower()
    search_term = f"%{normalized_query}%"

    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row

    rows = conn.execute(
        f"""
        SELECT DISTINCT d.*
        FROM {DOC_TABLE} d
        LEFT JOIN {ENT_TABLE} e ON d.doc_id = e.doc_id
        WHERE lower(d.cleaned_text) LIKE ?
           OR lower(d.title) LIKE ?
           OR lower(e.ent_text) LIKE ?
        """,
        (search_term, search_term, search_term),
    ).fetchall()

    return {
        "query": normalized_query,
        "matches": [dict(row) for row in rows],
        "count": len(rows),
    }


# @tool
# def add(a: int, b: int) -> int: #TODO Create search_news tool
#     """Adds `a` and `b`.

#     Args:
#         a: First int
#         b: Second int
#     """
#     return a + b


# @tool
# def divide(a: int, b: int) -> float: #TODO Create generate_brief tool
#     """Divide `a` and `b`.

#     Args:
#         a: First int
#         b: Second int
#     """
#     return a / b


# @tool
# def divide(a: int, b: int) -> float: #TODO Create build_timeline tool
#     """Divide `a` and `b`.

#     Args:
#         a: First int
#         b: Second int
#     """
#     return a / b


# Augment the LLM with tools
tools = [search_documents]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

SyntaxError: invalid syntax (3370832589.py, line 9)

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [ ]:
from langchain.messages import SystemMessage


def llm_call(state: dict):
    """LLM decides whether to call a tool or not"""

    return {
        "messages": [
            model_with_tools.invoke(
                [
                    SystemMessage(
                        content="You are a helpful assistant tasked deciding which tools to call."
                    )
                ]
                + state["messages"]
            )
        ],
        "llm_calls": state.get('llm_calls', 0) + 1
    }

In [ ]:
from langchain.messages import ToolMessage


def tool_node(state: dict):
    """Performs the tool call"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        tool = tools_by_name[tool_call["name"]]
        observation = tool.invoke(tool_call["args"])
        result.append(ToolMessage(content=observation, tool_call_id=tool_call["id"]))
    return {"messages": result}

In [ ]:
from typing import Literal
from langgraph.graph import StateGraph, START, END


def should_continue(state: MessagesState) -> Literal["tool_node", END]:
    """Decide if we should continue the loop or stop based upon whether the LLM made a tool call"""

    messages = state["messages"]
    last_message = messages[-1]

    # If the LLM makes a tool call, then perform an action
    if last_message.tool_calls:
        return "tool_node"

    # Otherwise, we stop (reply to the user)
    return END

In [ ]:
# Build workflow
agent_builder = StateGraph(MessagesState)

# Add nodes
agent_builder.add_node("llm_call", llm_call)
agent_builder.add_node("tool_node", tool_node)

# Add edges to connect nodes
agent_builder.add_edge(START, "llm_call")
agent_builder.add_conditional_edges(
    "llm_call",
    should_continue,
    ["tool_node", END]
)
agent_builder.add_edge("tool_node", "llm_call")

# Compile the agent
agent = agent_builder.compile()

# Show the agent
from IPython.display import Image, display
display(Image(agent.get_graph(xray=True).draw_mermaid_png()))

# Invoke
from langchain.messages import HumanMessage
messages = [HumanMessage(content="Add 3 and 4.")]
messages = agent.invoke({"messages": messages})
for m in messages["messages"]:
    m.pretty_print()